# DS-02 · Student Early-Warning System
**Team:** _add names_   |   **Date:** 30 September 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 50)

# Figures are saved to reports/figures (repo layout) so they can be used in the README and slides
FIG_DIR = Path("../reports/figures") if Path("../data").exists() else Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def find_data(name):
    """Look in data/ first; in Colab without the repo, show a Choose Files button."""
    for p in [Path("../data") / name, Path("data") / name, Path(name)]:
        if p.exists():
            return p
    try:
        from google.colab import files
        print(f"Please upload {name}")
        up = files.upload()
        return Path(list(up.keys())[0])
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the data/ folder")

path = find_data("student_performance.csv")
df_raw = pd.read_csv(path)
print("Loaded", path, "→ shape", df_raw.shape)

## 1. Problem statement
**Client:** a school's academic office wants to flag students likely to fail the final exam **right after the midterm**, while there is still time to arrange extra support.

**Business question:** Using only information available right after the midterm, which students are at risk of failing, and what are the warning signs?

- **Task:** classification. **Target:** `Result` (0 = fail, 1 = pass). **Positive class: at-risk (fail)**, the group we want to find.
- **Success metric:** **recall for the at-risk class**, with precision and F1 alongside.
- **Why recall:** missing an at-risk student (false negative) means they fail without help; giving extra help to a student who would have passed (false positive) costs little. So the model must find as many failing students as possible.

## 2. Data dictionary
| Column | Meaning | Type | Unit | Notes |
|---|---|---|---|---|
| StudentID | Student identifier | id | – | Not a feature |
| Gender | Gender | category | text | One-hot |
| Age | Age | number | years | |
| ParentEducation | Highest parental education | category | text | **126 missing (25%)** |
| StudyHoursPerWeek | Self-reported study time | number | hours/week | **18 missing** |
| AttendanceRate | Share of classes attended | number | % | **28 missing** |
| InternetAccess | Internet at home | category | Yes/No | One-hot |
| MidtermScore | Midterm exam score | number | points | Known right after the midterm |
| FinalScore | Final exam score | number | points | **Leakage: known only after the final** |
| Result | Pass / fail | binary | 1 pass, 0 fail | **Target** (= FinalScore ≥ 50) |

## 3. Data audit & cleaning

In [ ]:
df = df_raw.copy(); df.columns = df.columns.str.strip()
for col in ["Gender", "ParentEducation", "InternetAccess"]:
    df[col] = df[col].astype("string").str.strip().str.title()
if not pd.api.types.is_numeric_dtype(df["Result"]):
    df["Result"] = df["Result"].astype(str).str.strip().str.lower().map({"pass": 1, "fail": 0, "1": 1, "0": 0})
print("Shape:", df.shape)
print("\nMissing:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum(), "| Duplicate StudentID:", df["StudentID"].duplicated().sum())
print("\nResult:", df["Result"].value_counts().rename({0: "fail", 1: "pass"}).to_dict())
df.describe()

In [ ]:
for col in ["Age", "StudyHoursPerWeek", "AttendanceRate", "MidtermScore"]:
    q1, q3 = df[col].quantile([0.25, 0.75]); lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    print(f"{col:18s}: {((df[col] < lo) | (df[col] > hi)).sum()} outliers, range {df[col].min()}–{df[col].max()}")
df["ParentEducation"] = df["ParentEducation"].fillna("Unknown")
df["AtRisk"] = (df["Result"] == 0).astype(int)

**Decision log** (✍️ check the numbers against the outputs above)
| Issue | Decision | Reason |
|---|---|---|
| **ParentEducation: 126 missing (25%)** | New category **"Unknown"** | Dropping would lose a quarter of students; "unknown" may itself carry information |
| **StudyHoursPerWeek (18) & AttendanceRate (28) missing** | **Training-set median**, inside the pipeline | Few rows; median is robust to outliers; no test-set leakage |
| Gender, InternetAccess, ParentEducation are text | **One-hot** inside the pipeline | Models need numbers; no order between categories |
| Target Result | New column **AtRisk = 1 if fail** | Makes "fail" the positive class for recall |
| Class imbalance | **Stratified** split; recall/F1 instead of accuracy alone | So both classes appear in train and test in the same proportion |
| Outliers | _✍️ kept / removed, and why_ | |

## 4. Exploratory data analysis

In [ ]:
c = df["AtRisk"].map({1: "At-risk (fail)", 0: "Pass"}).value_counts()
plt.bar(c.index, c.values, color=["tab:red", "tab:green"][: len(c)]); plt.title("Class balance"); save_fig("01_class_balance"); c

✍️ **Insight:** _write one or two sentences from the output above._ _Note why accuracy alone would mislead with this balance._

In [ ]:
def pass_rate(col, fname, title, bands=True):
    g = pd.qcut(df[col], 4, duplicates="drop") if bands else df[col]
    r = df.groupby(g, observed=True)["Result"].mean()
    ax = r.plot.bar(ylim=(0, 1), rot=0, figsize=(8, 4))
    for i, v in enumerate(r): ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
    ax.set(title=title, ylabel="Pass rate", xlabel=""); save_fig(fname); return r.round(3)
pass_rate("AttendanceRate", "02_attendance", "Pass rate by attendance band (quartiles)")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
pass_rate("StudyHoursPerWeek", "03_study_hours", "Pass rate by weekly study hours (quartiles)")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
pass_rate("InternetAccess", "04_internet", "Pass rate by internet access", bands=False)

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
sns.boxplot(x=df["Result"].map({0: "Fail", 1: "Pass"}), y=df["MidtermScore"]); plt.title("Midterm score: fail vs pass"); plt.xlabel("")
save_fig("05_midterm"); df.groupby("Result")["MidtermScore"].describe().round(1)

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
pass_rate("ParentEducation", "06_parent_education", "Pass rate by parental education", bands=False)

✍️ **Insight:** _write one or two sentences from the output above._ _Is the "Unknown" group different from the others?_

## 5. Feature preparation

In [ ]:
# LEAKAGE CHECK: is FinalScore hiding the answer?
num = df.drop(columns=["StudentID"]).select_dtypes("number")
print("Correlation with Result:\n", num.corr()["Result"].drop(["Result", "AtRisk"]).sort_values(ascending=False).round(3))
agree = ((df["FinalScore"] >= 50).astype(int) == df["Result"]).mean()
print(f"\nRule 'FinalScore >= 50 → pass' matches Result for {agree:.1%} of students")
sns.stripplot(x=df["FinalScore"], y=df["Result"].map({0: "Fail", 1: "Pass"}), alpha=0.5, jitter=0.25)
plt.axvline(50, color="red", ls="--"); plt.title("Result is decided by FinalScore → leakage"); plt.ylabel(""); save_fig("07_leakage")

**Leakage verdict:** `Result` is simply `FinalScore ≥ 50`, and FinalScore is only known **after** the final exam, too late to help anyone. A model using it would score ~100% on paper and be useless in real life. **FinalScore is removed** (StudentID too, as it is an ID). ✍️ _Quote the match % from the output._

- **Features:** Age, StudyHoursPerWeek, AttendanceRate, MidtermScore (numeric) + Gender, ParentEducation, InternetAccess (one-hot).
- **Imputation and encoding** happen inside a pipeline fitted on training data only. **Scaling** is used only for Logistic Regression.

In [ ]:
num_cols, cat_cols = ["Age", "StudyHoursPerWeek", "AttendanceRate", "MidtermScore"], ["Gender", "ParentEducation", "InternetAccess"]
X, y = df[num_cols + cat_cols], df["AtRisk"]
def make_prep():
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)], verbose_feature_names_out=False)

## 6. Modelling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
def scores(name, yt, yp): return {"Model": name, "Accuracy": accuracy_score(yt, yp), "Precision (at-risk)": precision_score(yt, yp, zero_division=0),
                                  "Recall (at-risk)": recall_score(yt, yp), "F1 (at-risk)": f1_score(yt, yp)}
results = [scores("Baseline: everyone fails", y_test, np.ones(len(y_test), dtype=int))]
cv = StratifiedKFold(5, shuffle=True, random_state=42)
tuning = pd.DataFrame([{"max_depth": d, **{k: v.mean() for k, v in cross_validate(
    Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=d, random_state=42))]),
    X_train, y_train, cv=cv, scoring="f1", return_train_score=True).items() if "score" in k}} for d in range(2, 9)])
tuning.plot(x="max_depth", y=["train_score", "test_score"], marker="o", figsize=(7, 4), title="Tuning max_depth (CV F1, at-risk)")
plt.legend(["Train F1", "CV F1"]); save_fig("08_tuning")
best_depth = int(tuning.loc[tuning["test_score"].idxmax(), "max_depth"]); print("Best max_depth:", best_depth)

✍️ **Insight:** _write one or two sentences from the output above._ _What happens to train vs CV score at depth 8 (overfitting)?_

In [ ]:
tree = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=best_depth, random_state=42))]).fit(X_train, y_train)
pred_tree = tree.predict(X_test); results.append(scores(f"Decision Tree (depth {best_depth})", y_test, pred_tree))
for name, cw in [("Logistic Regression", None), ("Logistic Regression balanced", "balanced")]:
    lr = Pipeline([("prep", make_prep()), ("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=1000, class_weight=cw))]).fit(X_train, y_train)
    results.append(scores(name + " (stretch)", y_test, lr.predict(X_test)))
tb = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=best_depth, class_weight="balanced", random_state=42))]).fit(X_train, y_train)
results.append(scores(f"Decision Tree balanced (stretch)", y_test, tb.predict(X_test)))

## 7. Evaluation

In [ ]:
pd.DataFrame(results).set_index("Model")

✍️ **Result:** _Why does "everyone fails" get decent accuracy but is useless? Which model has the best recall/F1? How did class_weight='balanced' change recall?_

In [ ]:
cm = confusion_matrix(y_test, pred_tree, labels=[1, 0])
ConfusionMatrixDisplay(cm, display_labels=["At-risk", "Pass"]).plot(cmap="Blues"); plt.title("Confusion matrix – decision tree"); save_fig("09_confusion_matrix")
TP, FN, FP, TN = cm.ravel()
print(f"TP={TP}: at-risk students flagged → get help in time\nFN={FN}: at-risk students MISSED → fail without support (costly)")
print(f"FP={FP}: passing students flagged → extra help not needed (small cost)\nTN={TN}: passing students correctly left alone")

✍️ **Confusion matrix in school terms:** _explain each number for the academic office._

In [ ]:
small = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=3, random_state=42))]).fit(X_train, y_train)
names = small.named_steps["prep"].get_feature_names_out()
plt.figure(figsize=(22, 9)); plot_tree(small.named_steps["tree"], feature_names=names, class_names=["Pass", "At-risk"], filled=True, rounded=True, proportion=True, fontsize=10)
save_fig("10_tree"); print(export_text(small.named_steps["tree"], feature_names=list(names)))

In [ ]:
err = X_test.assign(Actual=y_test, Pred=pred_tree)
print("Missed at-risk students (FN):"); err[(err.Actual == 1) & (err.Pred == 0)].head(10)

✍️ **Error analysis:** _what do the missed students have in common (midterm, attendance)?_

## 8. Conclusion & recommendation
✍️ _Fill in from the tree and the results:_

**Three warning signs for the academic office** (from the depth-3 tree)
1. **Midterm score below __** → high risk: arrange support immediately.
2. **Midterm between __ and __ AND attendance below __%** → at risk: contact student and parents.
3. **Study hours below __ per week with a weak midterm** → at risk: add to a study-skills group.

- **Answer:** the tuned model finds __% of failing students (recall) with precision __, vs the baseline that flags everyone.
- **Recommendation:** run the model right after each midterm and give the flagged list to form teachers.
- **Limitations:** 500 students, one course dataset; self-reported study hours; 25% unknown parental education; a flag is a prompt for a conversation, not a label.
- **Next steps:** add assignment scores and attendance trends; re-check recall each term.

### Viva notes
1. **Why no FinalScore?** Result = FinalScore ≥ 50 and it is only known after the final → leakage.
2. **FP or FN worse?** FN: an at-risk student gets no help.
3. **Why does "everyone fails" get decent accuracy?** Accuracy = share of the majority class; it helps no one.
4. **Why this max_depth?** Best CV F1; deeper trees overfit (train ≫ CV).
5. **Read one path of the tree** as a rule (see section 8).